In [1]:
# !git clone https://github.com/mashug0/BAH-ISRO-RCAN
!pip install importlib

  Preparing metadata (setup.py) ... done
  Created wheel for importlib: filename=importlib-1.0.4-py3-none-any.whl size=5850 sha256=d674a405054e26d6f51c6ea7c0e9a086c9f0b2774c3f9615b8984a0cf0fa2df4
  Stored in directory: /root/.cache/pip/wheels/03/4a/6e/7c4a313549653a504574fa29f907139c752051ef05210df605
Successfully built importlib


In [2]:
import sys
import os
import importlib
# sys.path.append('/kaggle/input/isro-bah-2-files')
# sys.path.append('/kaggle/input/isro-bah-2-files/Data/Data')
sys.path.append('/kaggle/input/bah-train-final')
sys.path.append('/kaggle/input/models-bah')
sys.path.append('/kaggle/input/bah-models-files')
!ls /kaggle/input/bah-models-files/Model

Allignment.py	     fuckit.txt.txt  MainShyt.py	    NoiseEstimator.py
DeformConv2d.py      __init__.py     MultiComponentLoss.py
FeatureExtractor.py  liteRCAN.py     NoiseAwareTDAN.py


In [3]:
import torch
import torch.nn
import torch.nn.functional as F
from Dataset.Degrador import Degrador
from Dataset.CreateDataset import CreateDataset
from Model.MainShyt import RobustLiteDualRCAN
from Model.MultiComponentLoss import MultiComponentLoss

from torch.utils.data import DataLoader
from torch.utils.data import Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# class CreateDataset(Dataset):
#     def __init__(self, hr_dir, lr1_dir, lr2_dir):
#         self.hr_files = sorted([os.path.join(hr_dir, f) for f in os.listdir(hr_dir)])
#         self.lr1_files = sorted([os.path.join(lr1_dir, f) for f in os.listdir(lr1_dir)])
#         self.lr2_files = sorted([os.path.join(lr2_dir, f) for f in os.listdir(lr2_dir)])

#     def __len__(self):
#         return len(self.hr_files)

#     def __getitem__(self, idx):
#         hr_img = torch.load(self.hr_files[idx], weights_only=False)
#         lr1_img = torch.load(self.lr1_files[idx], weights_only=False)
#         lr2_img = torch.load(self.lr2_files[idx], weights_only=False)

#         return {'HR': hr_img, 'LR1': lr1_img, 'LR2': lr2_img}

In [4]:
# hr_count = len(os.listdir("/kaggle/input/isro-bah-2-files/Data/Data/Spacenet/HR"))
# lr1_count = len(os.listdir("/kaggle/input/isro-bah-2-files/Data/Data/Spacenet/LR1"))
# lr2_count = len(os.listdir("/kaggle/input/isro-bah-2-files/Data/Data/Spacenet/LR2"))

# print(f"HR: {hr_count}, LR1: {lr1_count} , LR2: {lr2_count}")
from torchvision import transforms
from PIL import Image
class CreateDataset(Dataset):
    def __init__(self, HR_dir, LR_dir, transform=None):
        self.HR_dir = HR_dir
        self.LR_dir = LR_dir
        self.transform = transform if transform else transforms.ToTensor()
        
        self.hr_files = sorted([
            f for f in os.listdir(self.HR_dir) 
            if f.endswith('.tif')
        ])
        
        # Verify files exist
        self.valid_indices = []
        for idx in range(len(self.hr_files)):
            hr_path = os.path.join(self.HR_dir, self.hr_files[idx])
            lr1_path = os.path.join(self.LR_dir, self.hr_files[idx].replace('.tif', '_0.tif'))
            lr2_path = os.path.join(self.LR_dir, self.hr_files[idx].replace('.tif', '_1.tif'))
            
            if all(os.path.exists(p) for p in [hr_path, lr1_path, lr2_path]):
                self.valid_indices.append(idx)
            else:
                print(f"Missing files for HR: {self.hr_files[idx]}")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        real_idx = self.valid_indices[idx]
        hr_path = os.path.join(self.HR_dir, self.hr_files[real_idx])
        lr1_path = os.path.join(self.LR_dir, self.hr_files[real_idx].replace('.tif', '_0.tif'))
        lr2_path = os.path.join(self.LR_dir, self.hr_files[real_idx].replace('.tif', '_1.tif'))

        hr = Image.open(hr_path).convert('L')
        lr1 = Image.open(lr1_path).convert('L')
        lr2 = Image.open(lr2_path).convert('L')

        return {
            'HR': self.transform(hr),
            'LR1': self.transform(lr1),
            'LR2': self.transform(lr2),
            'name': os.path.splitext(self.hr_files[real_idx])[0]
        }

dataset = CreateDataset(
    '/kaggle/input/bah-train-final/BAH/BAH/HR',
    '/kaggle/input/bah-train-final/BAH/BAH/LR',
)

train_len = int(len(dataset) * 0.8)
val_len = int(len(dataset) * 0.1)
test_len = len(dataset) - train_len - val_len

train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(dataset,
                                          [train_len, val_len, test_len],
                                          generator=torch.Generator().manual_seed(42))
print(train_len)


8642


In [ ]:
!pip install lpips

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 363.4/363.4 MB 209.9 MB/s eta 0:00:0100:01

In [ ]:
!pip install torchmetrics sk-video kornia piq

In [ ]:
import lpips
from torchmetrics.image import StructuralSimilarityIndexMeasure as SSIM
import torch.nn as nn
import torch
from tqdm import tqdm
from torch.amp import GradScaler,autocast
from kornia.losses import SSIMLoss

lpips_model = lpips.LPIPS(net='alex').to(device).eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = MultiComponentLoss(perceptual_model=lpips_model ,w_ssim=0.3, w_edge=0.05, w_align=0.05, w_perc=0.01).to(device)

# Metrics
def psnr(sr, hr, max_val=1.0):
    mse = torch.mean((sr - hr) ** 2)
    return 10 * torch.log10(max_val ** 2 / mse)

ssim_torch = SSIM().to(device)

def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    total_ssim_loss = 0
    total_l1_loss = 0
    
    # Initialize losses and scaler
    l1_loss = nn.L1Loss()
    ssim_loss = SSIMLoss(window_size=11, reduction='none')
    scaler = GradScaler(device='cuda')
    
    # Progress bar
    progress_bar = tqdm(loader, desc="Training", leave=False)
    
    for batch in progress_bar:
        lr = batch['LR1'].to(device, non_blocking=True)
        lr2 = batch['LR2'].to(device, non_blocking=True)
        hr = batch['HR'].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        
        with autocast(device_type="cuda",dtype=torch.float16):
            sr = model(lr, lr2)
            
            # Loss calculations
            l1_term = l1_loss(sr, hr) * 0.9
            ssim_term = ssim_loss(sr, hr).mean() * 0.1
            loss = l1_term + ssim_term
        
        # Mixed precision backprop
        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        # Update metrics
        total_loss += loss.item()
        total_l1_loss += l1_term.item()
        total_ssim_loss += ssim_term.item()
        
        # Update progress bar
        progress_bar.set_postfix({
            'loss': f"{loss.item():.4f}",
            'L1': f"{l1_term.item():.4f}",
            'SSIM': f"{ssim_term.item():.4f}"
        })

    return {
        'total_loss': total_loss / len(loader),
        'ssim_loss': total_ssim_loss / len(loader),
        'l1_loss': total_l1_loss / len(loader)
    }


def train_epoch_2(model, loader, optimizer):
    model.train()
    total_loss = 0
    # Initialize total_metrics (was missing in original code)
    total_metrics = {"recon": 0.0, "ssim": 0.0, "edge": 0.0, "align": 0.0, "perc": 0.0}

    # Updated AMP syntax for PyTorch 2.0+
    scaler = torch.amp.GradScaler()  # Fixed deprecation warning
    progress_bar = tqdm(loader, desc="Training", leave=False)

    for batch in progress_bar:
        lr1 = batch['LR1'].to(device, non_blocking=True)
        lr2 = batch['LR2'].to(device, non_blocking=True)
        hr = batch['HR'].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        
        # Updated autocast syntax
        with torch.amp.autocast(device_type='cuda', dtype=torch.float16):  # Fixed deprecation warning
            # Forward pass
            sr_out, align_loss = model(lr1, lr2)

            # Compute loss & metrics
            loss, batch_metrics = criterion(sr_out, hr, align_loss)
            
        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        # Update totals
        total_loss += loss.item()
        for k in total_metrics:
            total_metrics[k] += batch_metrics[k]

        # Update progress bar
        progress_bar.set_postfix({
            'Total': f"{loss.item():.4f}",
            'Recon': f"{batch_metrics['recon']:.4f}",
            'SSIM': f"{batch_metrics['ssim']:.4f}",
            'Edge': f"{batch_metrics['edge']:.4f}",
            'Align': f"{batch_metrics['align']:.4f}"
        })

    # Calculate averages
    return {
        'total_loss': total_loss / len(loader),
        **{k: v / len(loader) for k, v in total_metrics.items()}
    }



@torch.no_grad()
def validate(model, loader):
    model.eval()
    total_psnr = 0
    total_ssim = 0
    total_lpips = 0
    
    # Progress bar
    progress_bar = tqdm(loader, desc="Validation", leave=False)
    
    for batch in progress_bar:
        lr = batch['LR1'].to(device, non_blocking=True)
        lr2 = batch['LR2'].to(device, non_blocking=True)
        hr = batch['HR'].to(device, non_blocking=True)

        sr,_ = model(lr, lr2)
        
        # Calculate metrics
        batch_psnr = psnr(sr, hr)
        batch_ssim = ssim_torch(sr, hr)
        batch_lpips = lpips_model(sr, hr).mean()
        
        # Update totals
        total_psnr += batch_psnr.item()
        total_ssim += batch_ssim.item()
        total_lpips += batch_lpips.item()
        
        # Update progress bar
        progress_bar.set_postfix({
            'PSNR': f"{batch_psnr.item():.2f}",
            'SSIM': f"{batch_ssim.item():.4f}",
            'LPIPS': f"{batch_lpips.item():.4f}"
        })
    
    return {
        'PSNR': total_psnr / len(loader),
        'SSIM': total_ssim / len(loader),
        'LPIPS': total_lpips / len(loader)
    }

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim
import math
import torch
import torch.nn.functional as F
from lpips import LPIPS

def show_dual_input_reconstructions(model, dataset, n_samples=5, device='cuda', figsize=(25, 15)):
    """
    Enhanced visualization for dual-input models (LR1 + LR2 → HR)

    Args:
        model: Your trained dual-input model
        dataset: PyTorch dataset yielding (lr1, lr2, hr) triplets
        n_samples: Number of samples to display
        device: Device to run model on
        figsize: Figure size (width, height)
    """
    model.eval()
    lpips_model = LPIPS(net='alex').to(device).eval()

    # Initialize metrics
    metrics = {
        'PSNR': [],
        'SSIM': [],
        'LPIPS': []
    }

    # Create figure with subplots
    n_cols = 4  # LR1, LR2, Reconstructed HR, Ground Truth HR
    fig, axes = plt.subplots(n_samples, n_cols, figsize=figsize)
    if n_samples == 1:
        axes = axes[np.newaxis, :]

    # Get random samples
    indices = np.random.choice(len(dataset), size=n_samples, replace=False)

    with torch.no_grad():
        for i, idx in enumerate(indices):
            batch = dataset[idx]
            lr1 = batch['LR1'].unsqueeze(0).to(device)
            lr2 = batch['LR2'].unsqueeze(0).to(device)
            hr = batch['HR'].unsqueeze(0).to(device)

            # Forward pass
            sr = model(lr1, lr2)

            # Convert to numpy (H,W,C) for color, (H,W) for grayscale
            def to_numpy_img(tensor):
                img_np = tensor.squeeze().cpu().numpy()
                if img_np.ndim == 3 and img_np.shape[0] in [1, 3]: # Handle grayscale (1 channel) or color (3 channels)
                    img_np = img_np.transpose(1, 2, 0)
                    if img_np.shape[-1] == 1: # Squeeze the last dimension if it's 1 (grayscale)
                        img_np = img_np.squeeze(-1)
                return img_np


            lr1_np = to_numpy_img(lr1)
            lr2_np = to_numpy_img(lr2)
            hr_np = to_numpy_img(hr)
            sr_np = to_numpy_img(sr)


            # Clip to valid range
            for img in [lr1_np, lr2_np, hr_np, sr_np]:
                np.clip(img, 0, 1, out=img)

            # Calculate metrics
            mse = F.mse_loss(sr, hr).item()
            psnr_val = -10 * math.log10(mse) if mse > 0 else float('inf')
            # Adjust SSIM for grayscale vs color
            if hr_np.ndim == 2: # Grayscale
                ssim_val = ssim(hr_np, sr_np, data_range=1.0)
            else: # Color
                ssim_val = ssim(hr_np, sr_np, channel_axis=-1, data_range=1.0)

            lpips_val = lpips_model(2*sr-1, 2*hr-1).mean().item()

            metrics['PSNR'].append(psnr_val)
            metrics['SSIM'].append(ssim_val)
            metrics['LPIPS'].append(lpips_val)

            # Plot images
            titles = [
                f"LR1 Input\n(Size: {lr1_np.shape[:2]})",
                f"LR2 Input\n(Size: {lr2_np.shape[:2]})",
                f"Reconstructed HR\nPSNR: {psnr_val:.2f} dB\nSSIM: {ssim_val:.4f}",
                f"Ground Truth HR\nLPIPS: {lpips_val:.4f}"
            ]

            imgs = [lr1_np, lr2_np, sr_np, hr_np]
            for j, (img, title) in enumerate(zip(imgs, titles)):
                ax = axes[i,j]
                # Use cmap='gray' for grayscale images
                ax.imshow(img, cmap='gray' if img.ndim == 2 else None)
                ax.set_title(title, fontsize=10)
                ax.axis('off')

    plt.tight_layout()
    plt.show()

    # Print metrics
    print("\n=== Performance Summary ===")
    print(f"Average PSNR: {np.mean(metrics['PSNR']):.2f} dB (higher better)")
    print(f"Average SSIM: {np.mean(metrics['SSIM']):.4f} (1.0 perfect)")
    print(f"Average LPIPS: {np.mean(metrics['LPIPS']):.4f} (lower better)")
    print("\nIndividual Samples:")
    for i, idx in enumerate(indices):
        print(f"Sample {i+1}: "
              f"PSNR={metrics['PSNR'][i]:.2f} dB, "
              f"SSIM={metrics['SSIM'][i]:.4f}, "
              f"LPIPS={metrics['LPIPS'][i]:.4f}")

def show_single_sr_sample(model, dataset, device='cuda', figsize=(12, 5)):
    """
    Visualize one sample (LR1 input, SR output, HR GT) side-by-side.
    LR1 will be smaller, SR ≈ HR in resolution.
    """
    import matplotlib.pyplot as plt
    import numpy as np
    from skimage.metrics import structural_similarity as ssim
    import torch.nn.functional as F
    import math

    model.eval()
    idx = np.random.randint(len(dataset))
    batch = dataset[idx]

    lr1 = batch['LR1'].unsqueeze(0).to(device)
    lr2 = batch['LR2'].unsqueeze(0).to(device)
    hr = batch['HR'].unsqueeze(0).to(device)

    with torch.no_grad():
        sr = model(lr1, lr2)

    def to_numpy_img(t):
        img_np = t.squeeze().detach().cpu().numpy()
        if img_np.ndim == 3 and img_np.shape[0] in [1, 3]:
            img_np = np.transpose(img_np, (1, 2, 0))
            if img_np.shape[-1] == 1:
                img_np = img_np.squeeze(-1)
        return np.clip(img_np, 0, 1)

    lr1_np = to_numpy_img(lr1)
    sr_np = to_numpy_img(sr)
    hr_np = to_numpy_img(hr)

    # Metrics
    mse = F.mse_loss(sr, hr).item()
    psnr_val = -10 * math.log10(mse) if mse > 0 else float('inf')
    ssim_val = ssim(hr_np, sr_np, data_range=1.0) if hr_np.ndim == 2 else ssim(hr_np, sr_np, channel_axis=-1, data_range=1.0)

    # Plot
    fig, axes = plt.subplots(1, 3, figsize=figsize)
    titles = [
        f"LR1 Input\n{lr1_np.shape}",
        f"SR Output\nPSNR: {psnr_val:.2f} dB\nSSIM: {ssim_val:.4f}",
        f"Ground Truth HR\n{hr_np.shape}"
    ]
    imgs = [lr1_np, sr_np, hr_np]
    for ax, img, title in zip(axes, imgs, titles):
        ax.imshow(img, cmap='gray' if img.ndim == 2 else None)
        ax.set_title(title, fontsize=10)
        ax.axis('off')

    plt.tight_layout()
    plt.show()
def visualize_progress(model, dataset,device='cuda', n_samples=1):
    """
    Visualize progress during training (called every 5 epochs)
    
    Args:
        model: Current model state
        dataset: Validation dataset
        epoch: Current epoch number
        device: Device to use
        n_samples: Number of samples to show
    """
    model.eval()
    
    # Get random samples
    indices = np.random.choice(len(dataset), size=n_samples, replace=False)
    
    with torch.no_grad():
        for idx in indices:
            batch = dataset[idx]
            lr1 = batch['LR1'].unsqueeze(0).to(device)
            lr2 = batch['LR2'].unsqueeze(0).to(device)
            hr = batch['HR'].unsqueeze(0).to(device)
            
            # Generate super-resolved image
            sr,_ = model(lr1, lr2)
            
            # Convert to numpy arrays
            def to_numpy(tensor):
                img = tensor.squeeze().cpu().numpy()
                if img.ndim == 3 and img.shape[0] in [1, 3]:
                    img = img.transpose(1, 2, 0)
                    if img.shape[-1] == 1:
                        img = img.squeeze(-1)
                return np.clip(img, 0, 1)
            
            lr1_np = to_numpy(lr1)
            lr2_np = to_numpy(lr2)
            sr_np = to_numpy(sr)
            hr_np = to_numpy(hr)
            
            # Calculate metrics
            mse = F.mse_loss(sr, hr).item()
            psnr_val = -10 * math.log10(mse) if mse > 0 else float('inf')
            ssim_val = ssim(hr_np, sr_np, data_range=1.0) if hr_np.ndim == 2 else \
                      ssim(hr_np, sr_np, channel_axis=-1, data_range=1.0)
            lpips_val = lpips_model(2*sr-1, 2*hr-1).mean().item()
            
            # Create figure with size ratios
            fig = plt.figure(figsize=(12, 6))
            gs = fig.add_gridspec(2, 4, width_ratios=[1, 1, 2, 2])
            
            # Plot LR images (smaller)
            ax1 = fig.add_subplot(gs[0, 0])
            ax1.imshow(lr1_np, cmap='gray' if lr1_np.ndim == 2 else None)
            ax1.set_title(f"LR1 Input\n{lr1_np.shape[:2]}")
            ax1.axis('off')
            
            ax2 = fig.add_subplot(gs[0, 1])
            ax2.imshow(lr2_np, cmap='gray' if lr2_np.ndim == 2 else None)
            ax2.set_title(f"LR2 Input\n{lr2_np.shape[:2]}")
            ax2.axis('off')
            
            # Plot SR and HR (larger)
            ax3 = fig.add_subplot(gs[:, 2])
            ax3.imshow(sr_np, cmap='gray' if sr_np.ndim == 2 else None)
            ax3.set_title(f"SR Output\nPSNR: {psnr_val:.2f} dB\nSSIM: {ssim_val:.4f}\nLPIPS: {lpips_val:.4f}")
            ax3.axis('off')
            
            ax4 = fig.add_subplot(gs[:, 3])
            ax4.imshow(hr_np, cmap='gray' if hr_np.ndim == 2 else None)
            ax4.set_title(f"Ground Truth")
            ax4.axis('off')
            
            plt.tight_layout()
            plt.show()
            
            print(f"Sample Metrics - PSNR: {psnr_val:.2f} dB, SSIM: {ssim_val:.4f}, LPIPS: {lpips_val:.4f}")

In [ ]:
# !pip install tqdm

from piq import brisque
import numpy as np
import torch # Import torch
from torch.utils.data import DataLoader

def train_model(model , epochs = 100 , lr = 1e-4):
  train_loader = DataLoader(train_dataset , 
                            batch_size = 2 , 
                            shuffle = True, 
                            pin_memory=True , 
                            num_workers = 1,
                            persistent_workers=True
)
  val_loader = DataLoader(val_dataset , 
                          batch_size = 2 , 
                          shuffle = True,
                          pin_memory=True,
                          num_workers = 1,
                         persistent_workers=True)

  optimizer = torch.optim.Adam(model.parameters() , lr = lr)
  scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer , 'max' , patience = 5)
    
  best_psnr = 0
  for epoch in range(epochs):
      
    train_loss = train_epoch_2(model, train_loader, optimizer)
    metrics = validate(model, val_loader)
    scheduler.step(metrics['PSNR'])

    if metrics['PSNR'] > best_psnr:
        best_psnr = metrics['PSNR']
        torch.save(model.state_dict(), 'best_model.pth')

        # Consistent logging with lowercase keys
    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss: {train_loss['total_loss']:.4f} | "
          f"SSIM Loss: {train_loss['ssim']:.4f} | "  # Lowercase 'ssim'
          f"L1 Loss: {train_loss['recon']:.4f} | "   # 'recon' instead of 'Recon'
          f"Edge Loss: {train_loss['edge']:.4f} | "
          f"Alignment Loss: {train_loss['align']:.4f} | "
          f"Val PSNR: {metrics['PSNR']:.2f} dB | "
          f"SSIM: {metrics['SSIM']:.4f} | "
          f"LPIPS: {metrics['LPIPS']:.4f}")

    # blind_test(model , val_loader)
    if epoch%5==0:
      visualize_progress(model , test_dataset , device = device)
      torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, f"checkpoint_epoch_{epoch+1}.pth")


def blind_test(model , loader):
  niqe_scores = []
  model.eval()
  with torch.no_grad():
    for i, batch in enumerate(loader):
      lr = batch['LR1'].to(device)
      lr2 = batch['LR2'].to(device)
      hr = batch['HR'].to(device)

      sr = model(lr , lr2).cpu()
      # Handle grayscale vs color for BRISQUE calculation
      sr_np = sr.squeeze().cpu().numpy()
      if sr_np.ndim == 3 and sr_np.shape[0] in [1, 3]:
          sr_np = sr_np.transpose(1, 2, 0)
          if sr_np.shape[-1] == 1:
              sr_np = sr_np.squeeze(-1)

      # Ensure numpy array is in the correct format for brisque (H, W) or (H, W, C)
      if sr_np.ndim == 2:
          niqe_scores.append(brisque(torch.from_numpy(sr_np).unsqueeze(0).unsqueeze(0))) # Add batch and channel dim for piq
      elif sr_np.ndim == 3:
           niqe_scores.append(brisque(torch.from_numpy(sr_np).permute(2,0,1).unsqueeze(0))) # Permute to C, H, W and add batch dim for piq


  print(f"Average BRISQUE: {np.mean([score.item() for score in niqe_scores])}") # Extract scalar values from tensors

In [ ]:
%%javascript
// Keeps Kaggle session alive by simulating activity every 5 mins
function keepAlive() {
    console.log("Keeping session alive");
    document.querySelector('body').click();  // Simulate a click
}
setInterval(keepAlive, 5 * 60 * 1000);  // Run every 5 minutes

In [ ]:
from importlib import reload
import torch.nn as nn

import torch

torch.backends.cudnn.benchmark = True
model = RobustLiteDualRCAN(in_channels = 1 , n_features = 32 , n_rg = 5 , n_rcab = 5 , scale =2).to(device)
model = nn.DataParallel(model)

train_model(model)

In [ ]:
import numpy as np
from skimage.metrics import structural_similarity as ssim
import torch

dummy_hr = torch.rand(1, 3, 256, 256).to(device)
dummy_sr = torch.rand(1, 3, 256, 256).to(device)

# Convert tensors to numpy arrays for skimage ssim
dummy_hr_np = dummy_hr.detach().cpu().numpy()
dummy_sr_np = dummy_sr.detach().cpu().numpy()


print("Test SSIM:", ssim(dummy_hr_np.squeeze(), dummy_sr_np.squeeze(), channel_axis=0, data_range=1.0))  # Should be ~0.1 for random images